In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install auto-round --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound, SignRoundConfig
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForCausalLM, AutoProcessor


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.14.0+cu130
CUDA Available: True
CUDA Version: 13.0
GPU Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "dots-studio/dots.mocr"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Hint: A new version of huggingface_hub (2.2.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 22 files:   0%|                                | 0/22 [00:00<?, ?it/s]Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /workspace/local_model/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

Reconstructing (incomplete total...):

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/643 [00:00<?, ?it/s]

In [8]:
model

DotsOCRForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rot

In [9]:

# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1000,      # Optimization iterations per block; balances calibration time and accuracy
    group_size=32,   # Number of weights sharing one scale. Cannot go below 32 (vLLM's GPTQ kernel won't run it). Only use 16 if targeting NVFP4.
    sym=True,        # Symmetric quantization; enables faster compute and broader engine compatibility
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                      # Target pretrained model to be quantized
    tokenizer=tokenizer,              # Tokenizer used to process text calibration samples
    processor=processor,              # Processor for multimodal models (vision/audio), if present
    scheme="W4A16",                   # Quantization precision: 4-bit weights and 16-bit activations
    alg_configs=alg_config,           # Applies the algorithm-specific optimization settings defined above
    nsamples=512,                     # Total calibration dataset samples to maintain quality and avoid overfitting
    batch_size=4,                     # Batch size for calibration; kept < 8 to avoid slow execution/high memory overhead
    seqlen=4096,                      # Calibration context length; captures long-range dependency activations
    low_gpu_mem_usage=False,          # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    enable_torch_compile=True,        # Compiles forward passes using torch.compile to accelerate calibration
    quant_nontext_module=False,       # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision
)


In [10]:
# Quantize once and export to disk across all major loader schemas
ar.quantize_and_save(
    OUTPUT_BASE_DIR,
    # Serializes weights to 3 formats simultaneously to guarantee compatibility:
    # - 'auto_round': Reference AutoRound format (vLLM GPTQ/Marlin, Transformers / native loader)
    # - 'auto_gptq': AutoGPTQ tensor layout (vLLM GPTQ/Marlin, TGI)
    format="auto_round,auto_gptq",
    # Overwrites weights in-place to avoid extra VRAM allocation
    inplace=True
)

2026-10-13 16:50:02 INFO base.py L1365: `torch.compile` is enabled
2026-10-13 16:50:03 INFO orchestrator.py L594: start to cache block inputs
2026-10-13 16:50:03 INFO mllm.py L86: Using MLLM template: dots_ocr
2026-10-13 16:50:03 INFO mllm.py L125: Multimodal model with non-MLLM calibration dataset 'NeelNanda/pile-10k' and quant_nontext_module=False: using the standard text dataloader (vision/audio towers are not being quantized, so text-only calibration through the full-model forward is sufficient).
2026-10-13 16:50:03 INFO calib_dataset.py L1229: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
2026-10-13 16:50:40 INFO device.py L1560: 'peak_ram': 13.25GB, 'peak_vram': 5.77GB
2026-10-13 16:50:40 INFO orchestrator.py L626: caching done
Quantizing model.layers.0:   0%|          | 0/28 [00:01<?, ?it/s]quantized 7/7 layers in the block, loss iter 0: 1.325e-03 -> iter 946: 3.469e-04
2026-1

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

packing: 100%|██████████| 196/196 [00:04<00:00, 45.49it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-13 17:02:02 INFO device.py L1560: 'peak_ram': 22.91GB, 'peak_vram': 22.46GB


(DotsOCRForCausalLM(
   (model): Qwen2Model(
     (embed_tokens): Embedding(151936, 1536)
     (layers): ModuleList(
       (0-27): 28 x Qwen2DecoderLayer(
         (self_attn): Qwen2Attention(
           (q_proj): QuantLinear()
           (k_proj): QuantLinear()
           (v_proj): QuantLinear()
           (o_proj): QuantLinear()
         )
         (mlp): Qwen2MLP(
           (gate_proj): QuantLinear()
           (up_proj): QuantLinear()
           (down_proj): QuantLinear()
           (act_fn): SiLUActivation()
         )
         (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
         (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
       )
     )
     (norm): Qwen2RMSNorm((1536,), eps=1e-06)
     (rotary_emb): Qwen2RotaryEmbedding()
   )
   (lm_head): Linear(in_features=1536, out_features=151936, bias=False)
   (vision_tower): DotsVisionTransformer(
     (patch_embed): DotsViTPreprocessor(
       (patchifier): DotsPatchEmbed(
         (proj): Conv2d(3, 1536, ker

In [11]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:  # noqa: BLE001
        print(f"[Hub] ❌ Error uploading: {e}")

In [12]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [13]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-round-auto-gptq"),
        f"{base_name}-W4A16-AutoRound",
        hf_token,
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-gptq"),
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token,
    )

else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-w4g32/auto-round-auto-gptq to Vishva007/dots.mocr-W4A16-AutoRound...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/dots.mocr-W4A16-AutoRound

[Hub] Pushing ./AutoRound/local_model-w4g32/auto-gptq to Vishva007/dots.mocr-W4A16-AutoRound-GPTQ...
[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/dots.mocr-W4A16-AutoRound-GPTQ
